In [127]:
import os
import requests
import tiktoken
import numpy as np
import random
import torch
from torch import nn
import math

In [128]:
input_file_path = './data/tinyshakespeare/input.txt'

with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read()
n = len(data)
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

enc = tiktoken.get_encoding('gpt2')
train_ids = torch.tensor(enc.encode_ordinary(train_data), dtype=torch.long)
val_ids = torch.tensor(enc.encode_ordinary(val_data), dtype=torch.long)
print(f"train tokens: {len(train_ids):,}")
print(f"val tokens: {len(val_ids):,}")

train tokens: 301,966
val tokens: 36,059


In [327]:
class CausalSelfAttention(nn.Module):
    def __init__(self, T, d_m, h):
        super().__init__()
        self.d_k = int(d_m / h)
        self.d_v = int(d_m / h)
        self.h = h

        self.W_Q = nn.Parameter(torch.randn((d_m, self.d_k * self.h)))
        self.W_K = nn.Parameter(torch.randn((d_m, self.d_k * self.h)))
        self.W_V = nn.Parameter(torch.randn((d_m, self.d_v * self.h)))
        self.register_buffer("M", torch.triu(torch.ones(T, T) * -torch.inf, diagonal=1))
        self.W_attn_out = nn.Parameter(torch.randn((h * self.d_v, d_m)))


    def forward(self, x):
        Q = x @ self.W_Q
        K = x @ self.W_K
        V = x @ self.W_V
        Q = Q.reshape((x.shape[0], x.shape[1], self.h, self.d_k)).permute(0, 2, 1, 3)  # (B, T, h, d_k) -> (B, h, T, d_k)
        K = K.reshape((x.shape[0], x.shape[1], self.h, self.d_k)).permute(0, 2, 1, 3)  # (B, T, h, d_k) -> (B, h, T, d_k)
        V = V.reshape((x.shape[0], x.shape[1], self.h, self.d_v)).permute(0, 2, 1, 3)  # (B, T, h, d_k) -> (B, h, T, d_k)
        S = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        seq_len = x.shape[1]
        M = self.M[:seq_len, :seq_len].unsqueeze(0).unsqueeze(0)  # (1, 1, T, T)
        P = torch.softmax(S + M, dim=-1)
        O = P @ V
        O = O.permute(0, 2, 1, 3).reshape(x.shape[0], x.shape[1], self.h * self.d_v)  # (B, h, T, d_v) -> (B, T, h, d_v) -> (B, T, h * d_v)
        out = O @ self.W_attn_out
        return out

In [328]:
class TransformerBlock(nn.Module):
    def __init__(self, T, d_m, h, d_ff):
        super().__init__()
        self.attn = CausalSelfAttention(T, d_m, h)
        self.layer_norm1 = nn.LayerNorm(d_m)

        self.l1 = nn.Linear(d_m, d_ff)
        self.relu = nn.ReLU()
        self.l2 = nn.Linear(d_ff, d_m)

        self.layer_norm2 = nn.LayerNorm(d_m)

    
    def forward(self, x):
        x_attn = self.attn(x)
        x_add_norm1 = self.layer_norm1(x_attn + x) 
        x_ff = self.l2(self.relu(self.l1(x_add_norm1)))
        x_add_norm2 = self.layer_norm2(x_ff + x_add_norm1)
        return x_add_norm2

In [329]:
class ToyGPT(nn.Module):
    def __init__(self, T, vocab_size, L, h, d_m, d_ff):
        super().__init__()
        self.E = nn.Parameter(torch.randn((vocab_size, d_m)))
        self.P = nn.Parameter(torch.randn((T, d_m)))

        self.blocks = nn.ModuleList([TransformerBlock(T, d_m, h, d_ff) for _ in range(L)])

        self.proj = nn.Linear(d_m, vocab_size)


    def forward(self, x):
        seq_len = x.shape[1]
        x_emb = self.E[x] + self.P[:seq_len]
        x = x_emb
        for block in self.blocks:
            x = block(x)
        logits = self.proj(x)
        return logits

In [330]:
T = block_size = 32
vocab_size = enc.n_vocab
L = n_layer = 3
h = n_head = 4
d_m = n_embd = 96
d_k = int(d_m / h)
d_v = int(d_m / h)
d_ff = 384  # 4 * d_m
B = batch_size = 8

In [331]:
model = ToyGPT(T, vocab_size, L, h, d_m, d_ff)

In [134]:
Xs = []
Ys = []
for _ in range(batch_size):
    i = random.randrange(len(train_ids) - T)
    x = train_ids[i : i + T]
    y = train_ids[i + 1 : i + T + 1]
    Xs.append(x)
    Ys.append(y)
x_dummy_batched = torch.stack(Xs)

In [135]:
x_dummy_batched.shape

torch.Size([8, 32])

In [136]:
model(x_dummy_batched).shape

torch.Size([8, 4, 32, 32])
torch.Size([1, 1, 32, 32])
torch.Size([8, 4, 32, 32])
torch.Size([1, 1, 32, 32])
torch.Size([8, 4, 32, 32])
torch.Size([1, 1, 32, 32])


torch.Size([8, 32, 50257])

In [126]:
model

ToyGPT(
  (blocks): ModuleList(
    (0-2): 3 x TransformerBlock(
      (attn): CausalSelfAttention()
      (layer_norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
      (l1): Linear(in_features=96, out_features=384, bias=True)
      (relu): ReLU()
      (l2): Linear(in_features=384, out_features=96, bias=True)
      (layer_norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    )
  )
  (proj): Linear(in_features=96, out_features=50257, bias=True)
)

In [318]:
# overfit one batch
x = train_ids[:T].unsqueeze(0)
y = train_ids[1 : T + 1]

In [344]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

In [345]:
for i in range(1000):
    logits = model(x).squeeze(0)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f"loss: {loss.item()}")


loss: 0.6510487794876099
loss: 0.6510499119758606
loss: 0.6510437726974487
loss: 0.6510394811630249
loss: 0.6510295867919922
loss: 0.6510299444198608
loss: 0.6510238647460938
loss: 0.6510186195373535
loss: 0.6510132551193237
loss: 0.651009738445282
loss: 0.651005208492279
loss: 0.6509988903999329
loss: 0.6509920358657837
loss: 0.6509878635406494
loss: 0.6509813070297241
loss: 0.6509791612625122
loss: 0.6509731411933899
loss: 0.6509665846824646
loss: 0.65096116065979
loss: 0.650956928730011
loss: 0.6509528756141663
loss: 0.6509487628936768
loss: 0.6509400606155396
loss: 0.6509374976158142
loss: 0.6509318947792053
loss: 0.6509268879890442
loss: 0.650920033454895
loss: 0.65091872215271
loss: 0.6509115099906921
loss: 0.6509105563163757
loss: 0.6509013772010803
loss: 0.6508979201316833
loss: 0.6508897542953491
loss: 0.6508852243423462
loss: 0.6508819460868835
loss: 0.6508769989013672
loss: 0.6508709192276001
loss: 0.6508621573448181
loss: 0.6508626341819763
loss: 0.6508554816246033
loss: 0.

In [212]:
x_tmp = train_ids[:T].unsqueeze(0)
logits = model(x_tmp)
probs = torch.softmax(logits, dim=-1)

In [214]:
tokens = [enc.decode([int(i)]) for i in probs[0].argmax(dim=1)]

In [215]:
tokens

[' Citizen',
 '\n',
 '\n',
 'Before',
 ' we',
 ' proceed',
 ' any',
 ' further',
 ',',
 ' hear',
 ' me',
 ' speak',
 '.',
 '\n',
 '\n',
 'All',
 ':',
 '\n',
 'Spe',
 'ak',
 ',',
 ' speak',
 '.',
 '\n',
 '\n',
 'First',
 ' Citizen',
 ':',
 '\n',
 'You',
 ' are',
 ' all']

In [217]:
[enc.decode([int(i)]) for i in x.squeeze()]

['First',
 ' Citizen',
 ':',
 '\n',
 'Before',
 ' we',
 ' proceed',
 ' any',
 ' further',
 ',',
 ' hear',
 ' me',
 ' speak',
 '.',
 '\n',
 '\n',
 'All',
 ':',
 '\n',
 'Spe',
 'ak',
 ',',
 ' speak',
 '.',
 '\n',
 '\n',
 'First',
 ' Citizen',
 ':',
 '\n',
 'You',
 ' are']

In [227]:
# autoregressive generation

In [348]:
model.eval()

tokens = enc.encode("First")  # prompt
x_gen = torch.tensor(tokens).unsqueeze(0)

In [349]:
with torch.inference_mode():
    for _ in range(50):
        x_cond = x_gen[:, -T:]  # trunc
        logits = model(x_cond)  # (1, seq_len, vocab_size)
        next_token_logits = logits[0, -1]  # (vocab_size,)
        next_token = torch.argmax(next_token_logits).item()
        x_gen = torch.cat([x_gen, torch.tensor([[next_token]])], dim=1)
        print(enc.decode([next_token]), end="", flush=True)

 Citizen:
BeforeBefore,Before any any
 CitizenYou
FirstFirstFirstFirstFirstFirstFirstFirstFirstFirstFirstFirst,FirstFirst areFirstFirst areYou, speakYou Citizen Citizen CitizenYou CitizenYou Citizen::: Citizen Citizen Citizen Citizen